# CLUSTER

## Imports

Este conjunto de celdas son para descargar el importar las librerías, el dataset y hacer la separación de variables (numéricas v.s. categóricas).

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [4]:
X = pd.read_csv("trainset.csv")
y = pd.read_csv("trainset_target.csv")

print(X)

       edad_an  tamano_hogar       psu  ratio_pobreza  tiempo_seg_cond1  \
0    -0.188751      1.575278  1.021598       0.984181          0.144067   
1     0.575265      0.208042  1.021598      -0.578577          0.144067   
2    -0.049839      0.208042  1.021598       1.226568          0.144067   
3    -0.605487      1.575278 -0.978858       0.021012          0.144067   
4     1.200370      0.208042  1.021598       0.250642          0.144067   
...        ...           ...       ...            ...               ...   
1914  1.269826      0.891660 -0.978858       0.697144         -0.195830   
1915 -1.925152      0.891660 -0.978858       1.226568          0.144067   
1916  0.853090      0.891660  1.021598       1.226568          0.144067   
1917  0.714178      1.575278  1.021598       1.226568          0.144067   
1918 -0.258207      0.891660 -0.978858      -1.184545          0.144067   

      tiempo_seg_cond2  tiempo_seg_cond3  tiempo_seg_cond4     pulso  \
0             0.245103     

In [6]:
var_cat = []

for col in X:

    valores_unicos = set(X[col].dropna().unique())

    # Detecta columnas binarias reales
    if valores_unicos.issubset({0.0, 1.0}):
        var_cat.append(col)

var_num = [
    col for col in X.columns
    if col not in var_cat
]

print(f"Categóricas: {len(var_cat)}")
print(f"Numéricas: {len(var_num)}")

Categóricas: 23
Numéricas: 46


## K-means

Para este K-means vamos a utilizar únicamente las variables numéricas.

> Métrica empleada: Silhouette

In [7]:
def evaluar_kmeans_silhouette(df, variables, k_max=4, random_state=42):
    """
    Evalúa K-Means usando silhouette score.
    Se asume que las variables ya están estandarizadas.
    """

    X = df[variables].copy().dropna()

    resultados = []

    for k in range(2, k_max + 1):

        modelo = KMeans(
            n_clusters=k,
            random_state=random_state,
            n_init=20
        )

        labels = modelo.fit_predict(X)

        silhouette = silhouette_score(X, labels)

        resultados.append({
            "k": k,
            "silhouette": silhouette,
            "inercia": modelo.inertia_,
            "n_observaciones": X.shape[0]
        })

    return pd.DataFrame(resultados).sort_values(
        by="silhouette",
        ascending=False
    )

In [8]:
resultados = evaluar_kmeans_silhouette(
    df= X,
    variables = var_num,
    k_max=4
)

print(resultados)

   k  silhouette       inercia  n_observaciones
0  2    0.097371  82580.621353             1919
2  4    0.055724  76253.396798             1919
1  3    0.050019  78672.752543             1919


In [9]:
mejor_k = resultados.iloc[0]["k"]

modelo_final = KMeans(
    n_clusters=int(mejor_k),
    random_state=42,
    n_init=20
)

clusters = modelo_final.fit_predict(X)

df_cluster = X.copy()
df_cluster["cluster"] = clusters

In [10]:
print(df_cluster["cluster"].value_counts())

print(
    df_cluster.groupby("cluster").agg(
        ["mean", "median", "std"]
    )
)

cluster
1    1169
0     750
Name: count, dtype: int64
          edad_an                     tamano_hogar                      \
             mean    median       std         mean    median       std   
cluster                                                                  
0        0.039806  0.158529  0.958107     0.036682 -0.475576  1.033797   
1       -0.025538  0.158529  1.025985    -0.023534 -0.475576  0.977863   

              psu                     ratio_pobreza  ... raza_etnia_3.0  \
             mean    median       std          mean  ...            std   
cluster                                              ...                  
0       -0.029308 -0.978858  0.999610     -0.215883  ...       0.495500   
1        0.018803 -0.978858  1.000653      0.138505  ...       0.494577   

        raza_etnia_4.0                  raza_etnia_6.0                   \
                  mean median       std           mean median       std   
cluster                                          

<u>**Análisis Resultados**</u>

El dataset no presenta una estructura de clusters clara bajo el supuesto de K-Means y distancia euclídea. 

En todos los valores de k revisados podemos apreciar que el valor de la métrica es < 0.1, lo que implica que los individuos están muy solapados. Esto encaja con el nuestro dataset biomédico dado que los pacientes (en este caso de mayoría sana) no forman grupos "naturales" lo q complica al cluster generar grupos claros.

> Se recomienda buscar otros métodos

